In [7]:
import pandas as pd

In [8]:
### includes allegation


def read_allegations():

    df = pd.read_csv(
        "../data/input/fl-2023-original-complaint-offenses.csv",
        encoding="latin",
    )

    df = df.fillna("")

    df = df[~(df.offense_comments == "")]

    df = df[["complaint_nbr", "offense_comments"]]
    return df


dfa = read_allegations()


def read_actions():
    df = pd.read_csv(
        "../data/input/fl-2023-original-complaint-discipline.csv",
        encoding="latin",
    )
    df = df.fillna("")

    df = df[~(df.discipline_imposed == "")]

    df = df[["complaint_nbr", "discipline_imposed", "discipline_comments"]]
    return df


dfb = read_actions()


def read_complaint_status():
    df = pd.read_csv(
        "../data/input/fl-2023-original-complaints.csv", encoding="latin"
    )
    df.loc[:, "case_opened_date"] = df.case_opened_date.str.replace(
        r"^(.+) (\w+):(\w+):(\w+)", r"\1", regex=True
    )
    df.loc[:, "case_closed_date"] = df.case_opened_date.str.replace(
        r"^(.+) (\w+):(\w+):(\w+)", r"\1", regex=True
    )

    df = df.fillna("")

    df = df[~(df.case_opened_date == "")]

    df = df[
        ["complaint_nbr", "person_nbr", "case_opened_date", "case_closed_date"]
    ]
    return df


dfc = read_complaint_status()

df = pd.merge(dfa, dfb, on="complaint_nbr").merge(dfc, on="complaint_nbr")

df

,complaint_nbr,offense_comments,discipline_imposed,discipline_comments,person_nbr,case_opened_date,case_closed_date
0,1,MISDEAMEANOR,NC-Dismissed,,61793,11/11/1976,11/11/1976
1,4,FELONY,Rev,,70322,7/11/1980,7/11/1980
2,20,FELONY,Rev,,85601,10/24/1980,10/24/1980
3,23,MISDEAMEANOR,Rev,,6849,4/4/1980,4/4/1980
4,24,MISDEAMEANOR,Rev,,28478,1/1/1980,1/1/1980
...,...,...,...,...,...,...,...
12430,49944,Felony Battery - Cause Great Bodily Harm,Rev,,324096,12/14/2022,12/14/2022
12431,49967,"Worker's Compensation; More than $20,000 but l...",Rev,,315444,12/21/2022,12/21/2022
12432,50012,Trafficking Phenethylamines 10 grams or more,Rev,,505091,12/22/2022,12/22/2022
12433,50086,Benzodiazepines,Rev,,529107,2/1/2023,2/1/2023


In [9]:
def read_index():
    df = pd.read_csv(
        "../data/input/fl-2023-index-enhanced.csv", encoding="latin"
    )
    return df


def read_officers():
    df = pd.read_csv(
        "../data/input/fl-2023-original-officers.csv", encoding="latin"
    )
    return df


index = read_index()

officers = read_officers()


def clean_demo_data(df):
    df.loc[:, "sex"] = (
        df.sex_code.str.lower()
        .str.strip()
        .fillna("")
        .str.replace(r"^f$", "Female", regex=True)
        .str.replace(r"^m$", "Male", regex=True)
        .str.replace(r"(u|o)", "", regex=True)
    )

    df.loc[:, "race"] = (
        df.race_code.str.lower()
        .str.strip()
        .fillna("")
        .str.replace(r"^(his|h)$", "Hispanic", regex=True)
        .str.replace(r"^(wh|whi)$", "White", regex=True)
        .str.replace(r"^blk$", "Black", regex=True)
        .str.replace(r"^as$", "Asian", regex=True)
        .str.replace(r"(oth|na)", "", regex=True)
    )

    df = df[["person_nbr", "sex", "race"]]

    return df


officers = officers.pipe(clean_demo_data)

index = pd.merge(index, officers, on="person_nbr")
index = index[
    [
        "person_nbr",
        "first_name",
        "middle_name",
        "last_name",
        "suffix",
        "year_of_birth",
        "agency",
        "type",
        "start_date",
        "end_date",
        "separation_reason",
        "race",
        "sex",
    ]
]

index

/var/folders/r9/3_1rmy995xs_9z4vz66rsf9r0000gn/T/ipykernel_54534/4195223562.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,person_nbr,first_name,middle_name,last_name,suffix,year_of_birth,agency,type,start_date,end_date,separation_reason,race,sex
0,99996,Seborn,E,Blackburn,NaN,NaN,Pasco-Hernando State College,Instructor,1994-02-15,1994-02-15,Voluntary Separation (Not involving misconduct),,Male
1,99995,David,A,Thomas,NaN,NaN,Orange County Sheriff's Office,Instructor,2013-03-14,2021-01-08,Retired (Not involving misconduct),,Male
2,99995,David,A,Thomas,NaN,NaN,"Valencia College, Criminal Justice Institute",Instructor,2004-05-10,2013-03-14,Instructor Request for Change of Affiliation,,Male
3,99995,David,A,Thomas,NaN,NaN,"Valencia College, Criminal Justice Institute",Instructor,1999-10-29,2003-10-01,Failure to Meet Mandatory Retraining Requirement,,Male
4,99994,Eric,C,Herb,NaN,1960.0,Sarasota County Sheriff's Office,Correctional,1992-06-11,1994-08-29,Voluntary Separation (Not involving misconduct),White,Male
...,...,...,...,...,...,...,...,...,...,...,...,...,...
646417,100,Barry,J,Smith,NaN,1946.0,Winter Garden Police Department,Law Enforcement,1974-03-28,1978-02-13,Resigned/Retired (Historical Use Only),White,Male
646418,1,Lex,Lane,Chance,NaN,1972.0,Florida Department Of Law Enforcement,Instructor,2008-12-04,NaN,NaN,White,Male
646419,1,Lex,Lane,Chance,NaN,1972.0,Florida Department Of Law Enforcement,Law Enforcement,2008-02-07,NaN,NaN,White,Male
646420,1,Lex,Lane,Chance,NaN,1972.0,Jefferson County Sheriff's Office,Law Enforcement,2007-12-17,2008-02-07,Voluntary Separation (Not involving misconduct),White,Male


In [10]:
merged_df = pd.merge(df, index, on="person_nbr", how="left")

# Convert date columns to datetime
date_columns = [
    "case_opened_date",
    "case_closed_date",
    "start_date",
    "end_date",
]
for col in date_columns:
    merged_df[col] = pd.to_datetime(merged_df[col])

# Filter to keep only records where case_opened_date is between start_date and end_date
merged_df = merged_df[
    (merged_df["case_opened_date"] >= merged_df["start_date"])
    & (
        merged_df["case_opened_date"]
        <= merged_df["end_date"].fillna(pd.Timestamp.now())
    )
]

merged_df.loc[:, "year_of_birth"] = (
    merged_df["year_of_birth"].astype(str).str.replace(r"\.0", "", regex=True)
)

merged_df = merged_df.rename(
    columns={"agency": "agency_name", "offense_comments": "offense"}
)


def proper_case(df):
    df.loc[:, "offense"] = df.offense.str.title()
    df.loc[:, "discipline_imposed"] = df.discipline_imposed.str.title()
    df.loc[:, "discipline_comments"] = df.discipline_comments.str.capitalize()
    return df


merged_df = merged_df.pipe(proper_case)

merged_df

/var/folders/r9/3_1rmy995xs_9z4vz66rsf9r0000gn/T/ipykernel_54534/961163612.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1938' '1951' '1946' ... '1999' '1985' '2001']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged_df.loc[:, "year_of_birth"] = (


,complaint_nbr,offense,discipline_imposed,discipline_comments,person_nbr,case_opened_date,case_closed_date,first_name,middle_name,last_name,suffix,year_of_birth,agency_name,type,start_date,end_date,separation_reason,race,sex
2,20,Felony,Rev,,85601,1980-10-24,1980-10-24,Earl,J,Beagles,NaN,1938,Tallahassee Police Department,Law Enforcement,1967-01-01,1980-10-24,Misconduct (Historical Use Only),White,Male
5,27,Misdeameanor,Rev,,100597,1980-10-31,1980-10-31,Ralph,J,Dibiasi,NaN,1951,St. Pete Beach Police Department,Law Enforcement,1980-04-29,1982-01-29,Misconduct (Historical Use Only),White,Male
17,43,Misdeameanor,Nc-Dismissed,,84929,1981-01-03,1981-01-03,Ronald,L,Dunn,NaN,1946,Hialeah Gardens Police Department,Law Enforcement,1980-09-22,1983-04-06,Resigned/Retired (Historical Use Only),White,Male
22,86,Misdeameanor,Sp,,100755,1981-02-17,1981-02-17,Leif,G,Fernandez,NaN,1950,Miami-Dade Police Department,Law Enforcement,1974-12-26,1981-05-01,No Cause for Decertification (Historical Use O...,Hispanic,Male
27,121,Felony,Rev,,70285,1981-04-01,1981-04-01,Raymond,NaN,Morales,NaN,1942,Key West Police Department,Law Enforcement,1977-02-11,1982-03-11,Misconduct (Historical Use Only),White,Male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28672,49496,9 Counts,Sus/Pr,,500394,2022-09-29,2022-09-29,Brynn,Darshon,Harvey,NaN,1989,Largo Police Department,Law Enforcement,2020-01-21,NaT,NaN,Black,Male
28676,49621,"Submitting Inaccurate, Incomplete, Untruthful ...",Di,,174213,2022-10-19,2022-10-19,David,NaN,Archey,NaN,1966,Department Of Corrections,Correctional,2017-08-11,NaT,NaN,White,Male
28697,49923,Without Violence,Rev,,532826,2022-12-02,2022-12-02,Devarus,Brayshard,Weston,NaN,1999,Department Of Corrections,Correctional,2022-08-22,2022-12-29,"Terminated for Violating Ch. 943.13(4), FS or ...",Black,Male
28702,49967,"Worker'S Compensation; More Than $20,000 But L...",Rev,,315444,2022-12-21,2022-12-21,Brett,A,Hayden,NaN,1985,Northeast Florida Criminal Justice Center,Instructor,2013-07-16,2023-11-27,"Terminated for Violating Ch. 943.13(4), FS or ...",White,Male


In [11]:
merged_df.head(10).to_csv(
    "../data/output/florida-discipline_index_head.csv", index=False
)

In [12]:
merged_df.to_csv("../data/output/florida-discipline_index.csv", index=False)